Step 1: Import Everything

In [14]:
import os
from dotenv import load_dotenv

#langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from sentence_transformers import CrossEncoder
from langchain_community.retrievers import BM25Retriever

from langchain.agents import create_agent

In [15]:
load_dotenv()

True

In [16]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

Loading our Data

In [52]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

DATA INGESTION

In [53]:
text_loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = text_loader.load()

In [54]:
print(documents)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content="# Employee HR Policies\n\n## 1. Purpose and Scope\n\nThis Employee HR Policy document establishes the general rules, procedures, and expectations that apply to all employees of the organization. The purpose of these policies is to provide a consistent and transparent framework for workplace conduct, attendance, compensation, leave, performance management, professional development, workplace safety, and employee responsibilities.\n\nThese policies apply to full-time employees, part-time employees, interns, trainees, and temporary employees unless a specific policy states otherwise. Certain departments or roles may have additional requirements because of the nature of their work. Where a departmental policy conflicts with a general HR policy, the applicable departmental policy should be reviewed with Human Resources.\n\nEmployees are expected to read and understand the policies applicable to their role. Failure to comply

## Langchain Documents
why do we need documents, langchain process everything in form of documents

Documents:
Page Conttent -- the actual data
Metadata -- extra info about the data
(It is just 1 doc right now but working in real world we have large of docs and metadata help us to knwo that where this data is coming from)

In [55]:
len(documents)

1

In [56]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [57]:
print(len(documents[0].page_content))

21870


Splitting our data

In [58]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 50 
)
chunks = text_splitter.split_documents(documents)

In [59]:
print(chunks[0])
print(len(chunks))

page_content='# Employee HR Policies

## 1. Purpose and Scope

This Employee HR Policy document establishes the general rules, procedures, and expectations that apply to all employees of the organization. The purpose of these policies is to provide a consistent and transparent framework for workplace conduct, attendance, compensation, leave, performance management, professional development, workplace safety, and employee responsibilities.' metadata={'source': 'data\\hr_policy.txt'}
55


Create Embedding Model

In [19]:
embedding_model = JinaEmbeddings(
    model_name = "jina-embeddings-v5-text-small"
)

Store Embeddings in Vector Database

In [ ]:
vector_store = FAISS.from_documents(chunks, embedding_model)

In [62]:
print(vector_store.index.ntotal)#total chunks are stored in the vector store

55


In [63]:
query = "What type of leaves can an employee take?"

context = vector_store.similarity_search(query, k=10)

for i, doc in enumerate(context):
    print(f"\n--- CHUNK {i+1} ---")
    print(doc.page_content)


--- CHUNK 1 ---
Employees whose employment ends must return company property and comply with continuing obligations relating to confidentiality, intellectual property, and other applicable agreements.

## 27. Grievance Procedure

Employees who have concerns regarding workplace treatment, management decisions, compensation, working conditions, harassment, discrimination, or other employment matters may raise a grievance through the designated HR process.

--- CHUNK 2 ---
## 5. Public Holidays

The organization observes public holidays according to the applicable national and regional holiday calendar. Employees required to work on an official holiday may receive compensation, time off in lieu, or another applicable benefit according to company policy and applicable employment regulations.

The HR department will communicate the official holiday schedule at the beginning of each calendar year or whenever changes occur.

## 6. Compensation and Payroll

--- CHUNK 3 ---
In cases of unexpec

In [64]:
query = "What type of leaves can an employee get?"

semantic_context = vector_store.similarity_search(query, k=20)

for i, doc in enumerate(context):
    print(f"\n--- RESULT {i+1} ---")
    print(doc.page_content)


--- RESULT 1 ---
Employees whose employment ends must return company property and comply with continuing obligations relating to confidentiality, intellectual property, and other applicable agreements.

## 27. Grievance Procedure

Employees who have concerns regarding workplace treatment, management decisions, compensation, working conditions, harassment, discrimination, or other employment matters may raise a grievance through the designated HR process.

--- RESULT 2 ---
## 5. Public Holidays

The organization observes public holidays according to the applicable national and regional holiday calendar. Employees required to work on an official holiday may receive compensation, time off in lieu, or another applicable benefit according to company policy and applicable employment regulations.

The HR department will communicate the official holiday schedule at the beginning of each calendar year or whenever changes occur.

## 6. Compensation and Payroll

--- RESULT 3 ---
In cases of unex

In [65]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5
query = "What type of leaves can an employee get?"

bm25_context = bm25_retriever.invoke(query)
for i, doc in enumerate(bm25_context, start = 1):
    print(f"\n--- RESULT {i} ---")
    print(doc.page_content)


--- RESULT 1 ---
Annual leave should normally be requested in advance through the company's approved leave management system. Employees should provide reasonable notice when planning extended periods of leave so that managers can arrange appropriate workload coverage.

Sick leave may be taken when an employee is unable to work because of illness or a medical condition. The company may request supporting documentation when required by applicable policy or law, particularly for extended periods of absence.

--- RESULT 2 ---
Employees working remotely should maintain reliable communication during working hours. If an employee experiences an internet, electricity, hardware, or other technical problem that prevents them from working, they should inform their supervisor and provide an estimated recovery time.

## 4. Leave Policy

Employees may be eligible for annual leave, sick leave, casual leave, parental leave, and other forms of leave according to their employment agreement and applicab

## RRF

In [31]:
def rrf(results, k = 60):
    scores = {}
    for result in results:
        for i, doc in enumerate(result, start = 1):
            doc_id = doc.page_content
            scores[doc_id] = scores.get(doc_id,0) + 1/(k+i)

    rrf_results = sorted(
            scores.items(),
            key=lambda x: x[1], #Sort using the second item, which is the score in reverse
            reverse=True
        )

    return rrf_results


In [67]:
results = [semantic_context, bm25_context]
rrf_results = rrf(results)
for rank, (doc, score) in enumerate(rrf_results, start=1):
    print(f"\n--- RANK {rank} | SCORE {score:.4f} ---")
    print(doc)


--- RANK 1 | SCORE 0.0299 ---
Annual leave should normally be requested in advance through the company's approved leave management system. Employees should provide reasonable notice when planning extended periods of leave so that managers can arrange appropriate workload coverage.

Sick leave may be taken when an employee is unable to work because of illness or a medical condition. The company may request supporting documentation when required by applicable policy or law, particularly for extended periods of absence.

--- RANK 2 | SCORE 0.0298 ---
Employees working remotely should maintain reliable communication during working hours. If an employee experiences an internet, electricity, hardware, or other technical problem that prevents them from working, they should inform their supervisor and provide an estimated recovery time.

## 4. Leave Policy

Employees may be eligible for annual leave, sick leave, casual leave, parental leave, and other forms of leave according to their employm

## Data Retrievel Pipeline

Reranking

In [32]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [69]:
pairs = [(query, doc) for (doc, sc) in rrf_results]

scores = reranker.predict(pairs)

ranked_documents = sorted(zip([doc for doc, sc in rrf_results], scores),
     key=lambda x: x[1], reverse = True)

In [70]:
for index,(doc, score) in enumerate(ranked_documents, start = 1):
    print(f"\n--- RANKED RESULT {index} ---")
    print(f"Score: {score}")
    print(doc)


--- RANKED RESULT 1 ---
Score: 3.9003453254699707
Annual leave should normally be requested in advance through the company's approved leave management system. Employees should provide reasonable notice when planning extended periods of leave so that managers can arrange appropriate workload coverage.

Sick leave may be taken when an employee is unable to work because of illness or a medical condition. The company may request supporting documentation when required by applicable policy or law, particularly for extended periods of absence.

--- RANKED RESULT 2 ---
Score: 1.3041601181030273
Employees working remotely should maintain reliable communication during working hours. If an employee experiences an internet, electricity, hardware, or other technical problem that prevents them from working, they should inform their supervisor and provide an estimated recovery time.

## 4. Leave Policy

Employees may be eligible for annual leave, sick leave, casual leave, parental leave, and other f

In [26]:
llm = ChatGroq(
    model_name = "openai/gpt-oss-120b",
    temperature = 0.3,
)

In [27]:
def augment_prompt(context, query):
    PROMPT_TEMPLATE = f"""
    context:{context}
    Query: {query}"""
    return PROMPT_TEMPLATE


In [73]:
system_message = {"role":"system", "content": "You are an HR assistant and you will only answer the queries based on the context provided. If the answer is not in the context, applogize and say 'I don't know' and tell the user your purpose"}
query = "What is the procedure for resignation?"
semantic_docs = vector_store.similarity_search(query, k=20)
keyword_docs = bm25_retriever.invoke(query)
rrf_results = rrf([semantic_docs, keyword_docs])
pairs = [(query, doc)for doc, sc in rrf_results]
scores = reranker.predict(pairs)
ranked_documents = sorted(zip([doc for doc, sc in rrf_results], scores), key = lambda x: x[1], reverse = True)
context = [doc for doc,sc in ranked_documents[:3]]

prompt = augment_prompt(context, query)

In [74]:
user_message = {"role":"user", "content": prompt}
messages = [system_message, user_message]

response = llm.invoke(messages)

In [75]:
print(response.content)

Based on the provided policy, the resignation procedure is:

1. **Submit written notice** – Provide a formal resignation letter that complies with the notice period set out in your employment agreement and any applicable law.  
2. **Serve the notice period** – Continue to perform your duties professionally and help ensure an orderly hand‑over of your work.  
3. **Complete exit requirements** – During the notice period you must:
   - Return all company property (e.g., laptops, ID cards, keys).  
   - Fill out any required exit documentation.  
   - Transfer relevant files, responsibilities, and knowledge to designated colleagues or your manager.  
   - Cooperate with the organization’s exit procedures to facilitate a smooth transition.  

Following these steps satisfies the organization’s resignation process.


## AI Agent

what are the the three thing AI agent contains --

1 - LLM -BRAIN

2 - Tool - Vector database here

3 - Mwmory (not using here) whenever asked related to intenral hr policies it gives error

Tools

In [89]:
semantic_retriever = vector_store.as_retriever(search_kwargs={"k":20})#it creating ab object of retrievr class which then can be used with langchain interfaces of .invoke and get relevand documents -> just like . similarity_search
keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 5

def search_hr_policy(question:str) -> str:
    """    Search the HR policy database for information related to
    company policies, leave, benefits, working hours, remote work,
    and other HR-related questions.
    """
    semantic_matching_chunks = semantic_retriever.invoke(question)
    keyword_matching_chunks = keyword_retriever.invoke(question)

    #Hybrid Retrieval using RRF
    rrf_results = rrf([semantic_matching_chunks, keyword_matching_chunks])

    #Reranking using CrossEncoder
    pairs = [(question, doc) for doc, sc in rrf_results]
    ranked_scores = reranker.predict(pairs)
    ranked_documents = sorted(zip([doc for doc, sc in rrf_results], ranked_scores), key = lambda x: x[1], reverse = True)

    #Returning the top 3 relevant documents as context
    context = [doc for doc,sc in ranked_documents[:3]]
    
    return "\n\n".join(context)

# it is a tool now ->it shoul d have  a doc string for an llm to understand which tool to use , what it does and how to use it.


In [90]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt = """
    You are a friendly Hr assistant. 
    Always use the search hr_policy tool to look up facts 
    before answering. If the answer isn't in the search 
    results, say you dont know instead of guessing.
    """
)

In [91]:
response = hr_assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me about leave policies?"
            }
        ]
    }
)

SystemMessage but you cant access it

Human Message - iindex =0

AI Message -  

Tool Message - if tool provided

AI Message -  last index = -1

In [92]:
print(response["messages"][-1].content)

Here’s a quick overview of the company’s leave policies as outlined in the HR policy document:

### Types of Leave
| Leave Type | When It Can Be Used | Key Points |
|------------|--------------------|------------|
| **Annual Leave** | Planned vacations, personal time off, or any other non‑work‑related reason. | • Must be requested in advance through the approved leave‑management system.<br>• Provide reasonable notice, especially for extended periods, so managers can arrange coverage. |
| **Sick Leave** | When you are unable to work due to illness or a medical condition. | • You may be asked for supporting documentation (e.g., a doctor’s note) for longer absences or when required by policy/law. |
| **Casual Leave** | Short‑notice, unplanned personal matters (e.g., a family emergency). | • Typically used for brief, unexpected situations. Check the system for any specific limits. |
| **Parental Leave** | Birth, adoption, or care of a newborn/young child. | • Eligibility and duration follo

In [88]:
response

{'messages': [HumanMessage(content='Tell me about leave policies?', additional_kwargs={}, response_metadata={}, id='5cace9b1-d2cb-4224-b270-ef3c85d770eb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer about leave policies. Must use search_hr_policy tool to look up facts. So we should query "leave policies".', 'tool_calls': [{'id': 'fc_e2027091-0f9b-496e-af1e-7ff40ebc7bad', 'function': {'arguments': '{"question":"leave policies"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 182, 'total_tokens': 239, 'completion_time': 0.125678405, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.007107227, 'prompt_tokens_details': None, 'queue_time': 0.005138951, 'total_time': 0.132785632}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_76082d2bd0', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider':

## Web Loader

In [1]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
loader = WebBaseLoader(web_path="https://docs.langchain.com/oss/python/deepagents/rag")

In [3]:
documents = loader.load()

In [8]:
print(documents)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/rag', 'title': 'Retrieval Augmented Generation (RAG) with Deep Agents - Docs by LangChain', 'description': 'RAG patterns for Deep Agents, including skills-guided retrieval, rubric grading, and a tutorial that indexes LangChain docs, offloads chunks to the filesystem, and delegates analysis to subagents', 'language': 'en'}, page_content='Retrieval Augmented Generation (RAG) with Deep Agents - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDeep AgentsRetrieval Augmented Generation (RAG) with Deep AgentsOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep A

In [9]:
print(documents[0].metadata)

{'source': 'https://docs.langchain.com/oss/python/deepagents/rag', 'title': 'Retrieval Augmented Generation (RAG) with Deep Agents - Docs by LangChain', 'description': 'RAG patterns for Deep Agents, including skills-guided retrieval, rubric grading, and a tutorial that indexes LangChain docs, offloads chunks to the filesystem, and delegates analysis to subagents', 'language': 'en'}


In [13]:
print(documents[0].metadata["description"])

RAG patterns for Deep Agents, including skills-guided retrieval, rubric grading, and a tutorial that indexes LangChain docs, offloads chunks to the filesystem, and delegates analysis to subagents


In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 50 
)
chunks = text_splitter.split_documents(documents)

In [18]:
print(len(chunks))

124


In [21]:
from langchain_chroma import Chroma

In [34]:
vector_store = Chroma.from_documents(
    chunks,
    embedding_model    
)

In [36]:
query = "How to create an agent?"
system_context = vector_store.similarity_search(query, k=3)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5
keyword_context = bm25_retriever.invoke(query)
rrf_results = rrf([system_context, keyword_context])
pairs = [(query, doc)for doc, sc in rrf_results]
scores = reranker.predict(pairs)
ranked_documents = sorted(zip([doc for doc, sc in rrf_results], scores), key = lambda x: x[1], reverse = True)
context = [doc for doc,sc in ranked_documents[:3]]
# for i, doc in enumerate(context, start=1):
#     print(f"\n--- RESULT {i} ---")
#     print(doc.page_content)

In [37]:
system_message = {"role":"system", "content": "You are an HR assistant and you will only answer the queries based on the context provided. If the answer is not in the context, applogize and say 'I don't know' and tell the user your purpose"}
prompt = augment_prompt(context, query)
user_message = {"role":"user", "content": prompt}
messages = [system_message, user_message]

response = llm.invoke(messages)

In [38]:
print(response.content)

Based on the provided context, creating an agent involves the following steps:

1. **Add the starter code to `agent.py`** – the context mentions “Add this code to agent.py,” which is the initial step for setting up the agent’s implementation.

2. **Use the Deep Agents library** – you can create a basic (no‑custom‑tools) Deep Agent with:

```python
from deepagents import create_deep_agent
```

3. **Prepare a query** – for example, the context shows how to formulate a query using `HumanMessage` from LangChain:

```python
from langchain.messages import HumanMessage

EXAMPLE_QUERY = "How do I stream intermediate tool results from a subagent?"
```

4. **Instantiate the agent and run the query** – after creating the agent with `create_deep_agent`, you can pass the `HumanMessage` containing your question to the agent and let it generate a response.

These steps give you a minimal, functional agent that can process a user query. For more advanced functionality (custom tools, documentation sear

In [39]:
from datetime import datetime

In [47]:
print(datetime.now().strftime("%d/%m/%Y_%H:%M:%S"))

09/10/2026_00:00:59
